In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
#import plotly.offline as pyo
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns
import netCDF4 as nc
import xarray as xr
#import cartopy.crs as ccrs
#from cartopy import feature as cfa
import os
import plotly.io as pio
pio.renderers.default = 'notebook'

In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res


def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft

def line_fit(X,Y):
    P = np.polyfit(X,Y,1)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.05, np.max(X)+dx*0.05])
    Y_new = np.polyval(P, X_new)
    return X_new, Y_new, P

In [ ]:
#%run ../20260323_read_data.py
%run ../../../2026-04-01_Suess-Effect/20260401_read_data.py

df_sorted

In [ ]:
blue = "#4287f5"
orange = "#f58442"
green = "#42f5ad"
red = "#f54254"
yellow = "#e4f000"
purple = "#f000ec"
green2 = "#00f070"
blue2 = "#00c0f0"
pink = "#e37b7b"
colors = [blue, orange, green, red, yellow, purple, green2, blue2, pink]



fig = make_subplots(specs=[[{"secondary_y": True}]])

for i, path in enumerate(paths):
    df_iter = df_sorted[df_sorted['path_name']==path]
    fig.add_trace( go.Scatter(x=df_iter['age_final'], y=df_iter['d13c'], mode='markers', marker=dict(color=colors[i]), name=path, text=df_iter['Name']))
    path_ma = df_iter['d13c'].rolling(window_size, center=True).mean()
    fig.add_trace(go.Scatter(x=df_iter['age_final'].rolling(window_size, center=True).mean(), y=path_ma, mode='lines',marker=dict(color=colors[i]), showlegend=False))


fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

fig.update_layout(width=1600, height=1000, title_text="d13c", template='simple_white')
fig.show()

#fig.write_html('d18O_paths.html')

In [ ]:
df_sorted['age_final_month'] = df_sorted['age_dt'].dt.month
df_updated_monthly = df_sorted[['age_final_month', 'd18o_pdb', 'd18o', 'd13c']].groupby('age_final_month').mean()
df_updated_monthly

In [ ]:
morpho_data = pd.read_csv('../../../2026-01-08_Luc-Morphology/SeaLevelFromMorphology.csv', header=None)
cols = ['date', 'SPA_luc', 'SPLA_luc','SPA_fred', 'SPLA_fred']
morpho_data.columns = cols
morpho_data

morpho_raw = pd.read_excel('../20260317_agemodel/Morphology_Data_ageupdated.xlsx')
#scale_factor_y = 10./(57.69-75.96)
scale_factor_y = (-10.-90.)/(3.03-149.03)
scale_factor_x = -scale_factor_y
offset_y =  -60-scale_factor_y*(94.27)
morpho_raw.loc[:,'x_true'] = morpho_raw.loc[:,'x']*scale_factor_x
morpho_raw.loc[:,'y_true'] = morpho_raw.loc[:,'y']*scale_factor_y + offset_y
morpho_memo = morpho_raw.loc[~morpho_raw['x'].isna(),:].sort_values("age_final")
morpho_memo

In [ ]:
window_size1=12

fred = pd.read_excel('../20260309_freds-data/fred_age_model_v2.xlsx').dropna()
fred['age_dt'] = decimal_year_to_datetime(fred['calendar_year'])
fred['d18o_SMOW'] = 1.03091*fred['d18o'] + 30.91
fred['d13c_suess'] = fred['d13c'] - _suess_fn(fred['calendar_year'])
fred[['age_ma','d13c_ma', 'd13c_suess_ma','d18o_ma', 'd18o_SMOW_ma']] = fred[['calendar_year','d13c','d13c_suess','d18o','d18o_SMOW']].rolling(window_size, center=True).mean()


cols = ['calendar_year','d13c','d13c_suess','d18o','d18o_SMOW']
new_cols = ['age_ma2','d13c_ma2','d13c_suess_ma2','d18o_ma2','d18o_SMOW_ma2']
rename_dict = dict(zip(cols,new_cols))
right_roll = fred[cols].rolling(window_size1).mean().rename(columns=rename_dict)
left_roll = fred[cols].rolling(window_size1,center=True).mean().shift(window_size1).rename(columns=rename_dict)
fred.loc[:89, new_cols] = right_roll.loc[:89].to_numpy()
fred.loc[90:, new_cols] = left_roll.loc[90:].to_numpy()

fred[['d13c_std','d18o_std']] = fred[['d13c','d18o_SMOW']].rolling(10, center=True).std()
fred_resampled = fred.drop(fred.columns[4:], axis=1)
fred_resampled['age_dt'] = decimal_year_to_datetime(fred_resampled['calendar_year'])
fred_resampled.set_index('age_dt', inplace=True)
fred_resampled = fred_resampled.resample('30D').mean()
fred_resampled.interpolate(inplace=True)
fred_indexed = fred.set_index('age_dt')
fred['month'] = fred['age_dt'].dt.month
fred_monthly = fred.groupby('month').mean()
fred

In [ ]:
climate = pd.read_csv('../../../2026-01-15_Climate-Model/analysis/climate_df.csv')
climate['datetime'] = pd.to_datetime(climate['datetime'])
climate_month = climate.groupby(climate['datetime'].dt.month).mean()
climate_month

In [ ]:
tide = pd.read_csv('../../../2025-10-10_Fred-TideGauge/memo_edited.csv')
tide['age_dt'] = decimal_year_to_datetime(tide['Date'])#.dt.date
tide = tide.set_index('age_dt')
tide

In [ ]:
enso_phase = pd.read_csv('../../../ENSO_Phase_Classification.csv')
enso_phase.columns=['Year','Phase']
enso_phase['Date'] = decimal_year_to_datetime(enso_phase['Year'])
#enso_phase.set_index('Date', inplace=True)
enso_phase['Phase'] = enso_phase['Phase'].str.replace('Warm Phase', 'W').str.replace('Cold Phase', 'C').str.replace('Neutral Phase','N')


enso = pd.read_csv("../../../ENSO_Time_Series.csv", parse_dates=[0])
#enso = enso.set_index("Date")
enso['day_of_year'] = enso['Date'].dt.dayofyear
enso['is_leap'] = enso['Date'].dt.is_leap_year
enso['total_days'] = np.where(enso['is_leap'], 366, 365)
enso['decimal_year'] = enso['Date'].dt.year + (enso['day_of_year'] - 1) / enso['total_days']


margin = 0.5
enso['phase'] = 'N'
enso.loc[enso['ENSO']>=margin,'phase'] = "C"
enso.loc[enso['ENSO']<=-margin,'phase'] = "W"
enso['phase_group'] = (enso['phase'] != enso['phase'].shift()).cumsum()
blocks = enso.groupby('phase_group').agg(
    start=('Date', 'first'),
    end=('Date', 'last'),
    value=('phase', 'first')
).reset_index(drop=True)

enso['Date'] = enso['Date'] + pd.offsets.MonthEnd(0)
enso.set_index('Date', inplace=True, drop=False)

ENSO_match = np.array([[1982.2, 1984.1],
                       [1986.75, 1988.3],
                       [1992.1, 1992.96],
                       [1997.2, 1998.8]])

enso_fft_freq, enso_fft = FFT_res(enso['ENSO'], 30)


enso


In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))


ax.plot(np.arange(1, 13), climate_month['synthetic_d18o'], '--o',color='blue',label='d18o climate')
ax.plot(np.arange(1, 13), df_updated_monthly['d18o_pdb']*2+10.2, '-^', color='steelblue', label='d18o obs')
ax.legend(loc='lower right')

ax.set_xlabel('Month of Year')
ax.set_ylabel(r'$\delta^{18}$O')
ax.set_xticks(np.arange(1, 13))


ax2 = ax.twinx()
#ax2.plot(climate_monthly['d13c'], color='green')
ax2.plot(np.arange(1, 13), df_updated_monthly['d13c'],  '-^',color='green',label='d13c obs')
#ax2.plot(df_updated_monthly['d18o_pdb']/df_updated_monthly['d18o_pdb'].max(), '--', color='blue', label='model')
ax2.set_ylabel(r'$\delta^{13}$C')
ax2.legend(loc='upper left')

fig.savefig('monthly_climate_model.pdf')

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(16,4))

ax = axes[0]
ax.plot(np.arange(1, 13), climate_month['synthetic_d18o'], '--o',color='steelblue', label=r'synthetic $\delta^{18}$O')
ax.plot(np.arange(1, 13), df_updated_monthly['d18o_pdb']*2+10.3, '-^', color='blue', label='d18o obs')
#ax.plot(np.arange(1, 13), fred_monthly['d18o']*2+10.8, '--s', color='cyan', label='Preliminary results')

ax.set_ylabel(r'$\delta^{18}$O')
ax.set_xlabel('Months')
ax.set_xticks(np.arange(1,13))
ax.set_title(r'Climate Average for $\delta^{18}$O')
ax.legend()
#fig.savefig('climate_average.pdf')
ax.set_xticks(np.arange(1, 13))

ax = axes[1]
ax.plot(np.arange(1, 13), climate_month['SSS'], '^-', color='#46bd9b', label='SSS')
ax2 = ax.twinx()
ax2.plot(np.arange(1, 13), climate_month['SST'], 'v-', color='#F54927', label='SST')
ax.set_xticks(np.arange(1, 13))
ax.legend(loc='lower right')
ax2.legend(loc='lower left')
ax.set_xlabel('Months')


ax = axes[2]
#ax2 = ax.twinx()
#ax2.plot(climate_monthly['d13c'], color='green')
ax.plot(np.arange(1, 13), df_updated_monthly['d13c'],  '-^',color='green',label='d13c obs')
#ax.plot(np.arange(1, 13), fred_monthly['d13c'],  '-^',color='green',label='Preliminary results')
#ax2.plot(df_updated_monthly['d18o_pdb']/df_updated_monthly['d18o_pdb'].max(), '--', color='blue', label='model')
ax.set_ylabel(r'$\delta^{13}$C')
ax.legend(loc='upper left')
ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")
ax.set_xlabel('Months')
ax.set_xticks(np.arange(1, 13))

fig.savefig('climate_isotpe_monthly.png')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), subplot_kw={'projection': 'polar'})

ax = axes[0]

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

theta = np.linspace(0, 2*np.pi, 12, endpoint=False)
width = 2*np.pi/12 * 0.9  # 90% of bin width (small gap)

rmin, rmax = -0.21 , 0.3
vals = climate_month['synthetic_d18o'].values

# Make bars start at rmin, and only show the variation within [rmin, rmax]
heights = np.clip(vals, rmin, rmax) - rmin

ax.bar(theta, heights, width=width, bottom=rmin,
       color='#1A388A', alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_rlim(rmin, rmax)

ax.set_rticks([])
ax.set_xticks(theta)
ax.set_xticklabels(months)

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)

ax.set_title(r'Climate based $\delta^{18}$O')

ax = axes[1]

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

theta = np.linspace(0, 2*np.pi, 12, endpoint=False)
width = 2*np.pi/12 * 0.9  # 90% of bin width (small gap)

rmin, rmax = -5.2, -4.95
vals = df_updated_monthly['d18o_pdb'].values

# Make bars start at rmin, and only show the variation within [rmin, rmax]
heights = np.clip(vals, rmin, rmax) - rmin

ax.bar(theta, heights, width=width, bottom=rmin,
       color='steelblue', alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_rlim(rmin, rmax)

ax.set_rticks([])
ax.set_xticks(theta)
ax.set_xticklabels(months)

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)

ax.set_title(r'Observed $\delta^{18}$O')


ax = axes[2]


rmin, rmax = -1.45, -1.18
vals = df_updated_monthly['d13c'].values

# Make bars start at rmin, and only show the variation within [rmin, rmax]
heights = np.clip(vals, rmin, rmax) - rmin

ax.bar(theta, heights, width=width, bottom=rmin,
       color='green', alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_rlim(rmin, rmax)

ax.set_rticks([])
ax.set_xticks(theta)
ax.set_xticklabels(months)

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)

ax.set_title(r'Observed $\delta^{13}$C')

plt.suptitle('Matching Peak Age Model', y= 1.05 , fontsize=16)


#fig.savefig('d13c_d18o_monthly_means.pdf',bbox_inches='tight' )
#fig.savefig('d13c_d18o_monthly_means.png',bbox_inches='tight' )

In [ ]:
fred.columns
#fred['calendar_year']

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])

#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o']-df_sorted['d18o'].mean(),  mode="markers", marker_color="#3492eb",  name='d18O', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled.index.values, y=df_resampled['d18o_ma']-df_resampled['d18o_ma'].mean(), mode="lines", line_color='#134b80', name='average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled.index.values, y=df_resampled['d18o_ma2']-df_resampled['d18o_ma2'].mean(), mode="lines", line_color='navy', name='average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o_no_tectonic'] ,  mode="markers", marker_color="#3492eb",  name='d18O', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o_notec_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d18o'] , mode="markers", line_color='#3492eb',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d18o_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d18o_SMOW'] , mode="markers", line_color='#3492eb',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['age_ma'], y=fred['d18o_SMOW_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)


fig.add_trace(go.Scatter(x=climate['age_absolute'], y= climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)



#fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[25.2], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
#                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[25.2], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
#                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[25.2], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
#                        name='M8.1 EQ 2007 Apr'))





fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O-AGEMODELTEST.html')


In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="markers", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)
#fig.add_trace(go.Scatter(x=enso['decimal_year'], y= enso['ENSO'] , mode="lines", line_color='gray', name='Sea Level', line_width=2), secondary_y=True)




fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.1 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[1959+(31+28+31+30+31+30+31+17)/365], y=[-1.395], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.2 EQ 1959 Aug')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=882668&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))





fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= 'Sea Level [mm]', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="markers", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d13c'] , mode="markers", line_color='cyan',opacity=0.7, name='frede'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d13c_ma'] , mode="lines", line_color='blue',opacity=0.7, name='frede'), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)
fig.add_trace(go.Scatter(x=enso['decimal_year'], y= enso['ENSO'] , mode="lines", line_color='gray', name='Sea Level', line_width=2), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= 'Sea Level [mm]', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:
fred.columns

In [ ]:
z

In [ ]:
df_sorted_indexed = df_sorted.set_index('age_dt')
primary_font = 15
secondary_font = 12

fig, ax = plt.subplots(1, figsize=(12,6))

ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d13c'], ".", color='lightgreen', alpha=0.7, label='main sampling' )
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d13c_sg_trend'],  color='darkgreen', lw=2.5, label='filtered' )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['calendar_year'], fred['d13c'], ".", color='#40E0D0', alpha=0.7, label='prelimanary sampling'  )
#ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

#plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
plt.axvline(2007+(31+29+31)/365, color='#EE27F5', label='M8.1 EQ 2007 Apr')



#date1 = '1936-04-19'
date1 = '1944-08-01'
#date2 = '1944-06'
date2 = '1950-11'
date2a = '1959-01'
#date3 = '1966-02'
date3 = '1970-10'
date4 = '1979-02'
date5 = '1992-06'
date6 = '2007-01'
date7 = '2010-01'

                   ###########  1
x, y, p= line_fit(df_sorted_indexed.loc[:date1,'age_final'].values, 
               df_sorted_indexed.loc[:date1,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean(), y.mean()-0.2, f'{p[0]:.3f}', color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  2
x, y, p= line_fit(df_sorted_indexed.loc[date1:date2,'age_final'].values, 
               df_sorted_indexed.loc[date1:date2,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '-', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


x, y, p= line_fit(df_sorted_indexed.loc[date2:date2a,'age_final'].values, 
               df_sorted_indexed.loc[date2:date2a,'d13c'].values)
ax.plot(x, y, linestyle='dashed', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '--', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-2, y.mean()-0.2, f'{p[0]:.3f}', color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  3
x, y, p= line_fit(df_sorted_indexed.loc[date2a:date3,'age_final'].values, 
               df_sorted_indexed.loc[date2a:date3,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '-', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  4
x, y, p= line_fit(df_sorted_indexed.loc[date3:date4,'age_final'].values, 
               df_sorted_indexed.loc[date3:date4,'d13c'].values)
ax.plot(x, y, linestyle='dashed', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '--', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  5
x, y, p= line_fit(df_sorted_indexed.loc[date4:date5,'age_final'].values, 
               df_sorted_indexed.loc[date4:date5,'d13c'].values)
ax.plot(x, y, linestyle='dashed', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '--', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()-0.2, f'{p[0]:.3f}', color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  6
x, y, p= line_fit(df_sorted_indexed.loc[date5:date6,'age_final'].values, 
               df_sorted_indexed.loc[date5:date6,'d13c'].values)
ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+0.5, y.mean()+0.2, f'{p[0]:.3f}', color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  7
x, y, p= line_fit(fred_indexed.loc[:date6,'calendar_year'].values, 
               fred_indexed.loc[:date6,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-1, y.mean()-0.4, f'{p[0]:.3f}', color='#178C81', bbox={'edgecolor':'#178C81' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})



                   ###########  8
x, y, p= line_fit(fred_indexed.loc[date6:date7,'calendar_year'].values, 
               fred_indexed.loc[date6:date7,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()+0.2, f'{p[0]:.3f}', color='#0A3D38', bbox={'edgecolor':'#0A3D38' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

for d in [date1, date2, date2a, date3, date4, date5, date6, date7]:
    plt.axvline(x=datetime_to_decimal_year(pd.to_datetime( d)),
               #ymin=-3.5, ymax=-1,linestyle='dashed',color='gray', alpha=0.6)
                linestyle='dashed',color='gray', alpha=0.6)
ax.set_ylabel(r'$\delta^{13}$C [PDB]', fontdict={'fontsize': 15})
ax.set_xlabel('Year', fontdict={'fontsize': 15})

ax.set_ylim(-3, 0.5)
ax.set_xlim(1936, 2012)

#ax.grid()
#ax.legend(loc='lower left')
#ax.legend(bbox_to_anchor=(1, 1))
ax.legend(
    #loc=(0.56, 0.6),
    loc='lower left',
    #bbox_to_anchor=(0.5, -0.15),  # Centered below the plot
    #ncol=2,                       # 3 columns
    frameon=True,
    facecolor='white',
    #columnspacing=1.5,
    #handletextpad=0.4,
    fontsize=secondary_font*0.75
)

ax.tick_params(axis='both', labelsize=secondary_font*0.85)


#fig.savefig('d13C_trens.pdf')
#fig.savefig('d13C_trens.png')

In [ ]:
df_sorted_indexed = df_sorted.set_index('age_dt')
primary_font = 15
secondary_font = 12

fig, ax = plt.subplots(1, figsize=(12,6))

ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d18o'], ".", color='#3492eb', alpha=0.7, label='main sampling' )
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d18o_sg_trend'],  color='#0D0454', lw=2.5, label='filtered' )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['calendar_year'], fred['d18o_SMOW'], ".", color='#40E0D0', alpha=0.7, label='prelimanary sampling'  )
#ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

#plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
plt.axvline(2007+(31+29+31)/365, color='#EE27F5', label='M8.1 EQ 2007 Apr')



#date1 = '1936-04-19'
date1 = '1944-08-01'
#date2 = '1944-06'
date2 = '1950-11'
date2a = '1959-01'
#date3 = '1966-02'
date3 = '1970-10'
date4 = '1979-02'
date5 = '1992-06'
date6 = '2007-01'
date7 = '2010-01'

                   ###########  1
x, y, p= line_fit(df_sorted_indexed.loc[:date1,'age_final'].values, 
               df_sorted_indexed.loc[:date1,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean(), y.mean()-0.2, f'{p[0]:.3f}', color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  2
x, y, p= line_fit(df_sorted_indexed.loc[date1:date2,'age_final'].values, 
               df_sorted_indexed.loc[date1:date2,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '-', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


x, y, p= line_fit(df_sorted_indexed.loc[date2:date2a,'age_final'].values, 
               df_sorted_indexed.loc[date2:date2a,'d18o'].values)
ax.plot(x, y, linestyle='dashed', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '--', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-2, y.mean()-0.2, f'{p[0]:.3f}', color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  3
x, y, p= line_fit(df_sorted_indexed.loc[date2a:date3,'age_final'].values, 
               df_sorted_indexed.loc[date2a:date3,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '-', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  4
x, y, p= line_fit(df_sorted_indexed.loc[date3:date4,'age_final'].values, 
               df_sorted_indexed.loc[date3:date4,'d18o'].values)
ax.plot(x, y, linestyle='dashed', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '--', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  5
x, y, p= line_fit(df_sorted_indexed.loc[date4:date5,'age_final'].values, 
               df_sorted_indexed.loc[date4:date5,'d18o'].values)
ax.plot(x, y, linestyle='dashed', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '--', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()-0.2, f'{p[0]:.3f}', color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  6
x, y, p= line_fit(df_sorted_indexed.loc[date5:date6,'age_final'].values, 
               df_sorted_indexed.loc[date5:date6,'d18o'].values)
ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+0.5, y.mean()+0.2, f'{p[0]:.3f}', color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  7
x, y, p= line_fit(fred_indexed.loc[:date6,'calendar_year'].values, 
               fred_indexed.loc[:date6,'d18o_SMOW'].values)
ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-1, y.mean()-0.4, f'{p[0]:.3f}', color='#178C81', bbox={'edgecolor':'#178C81' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})



                   ###########  8
x, y, p= line_fit(fred_indexed.loc[date6:date7,'calendar_year'].values, 
               fred_indexed.loc[date6:date7,'d18o_SMOW'].values)
ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()+0.2, f'{p[0]:.3f}', color='#0A3D38', bbox={'edgecolor':'#0A3D38' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

for d in [date1, date2, date2a, date3, date4, date5, date6, date7]:
    plt.axvline(x=datetime_to_decimal_year(pd.to_datetime( d)),
               #ymin=-3.5, ymax=-1,linestyle='dashed',color='gray', alpha=0.6)
                linestyle='dashed',color='gray', alpha=0.6)
ax.set_ylabel(r'$\delta^{18}$O [PDB]', fontdict={'fontsize': 15})
ax.set_xlabel('Year', fontdict={'fontsize': 15})

#ax.set_ylim(-3, 0.5)
ax.set_xlim(1936, 2012)

#ax.grid()
#ax.legend(loc='lower left')
#ax.legend(bbox_to_anchor=(1, 1))
ax.legend(
    #loc=(0.56, 0.6),
    loc='lower left',
    #bbox_to_anchor=(0.5, -0.15),  # Centered below the plot
    #ncol=2,                       # 3 columns
    frameon=True,
    facecolor='white',
    #columnspacing=1.5,
    #handletextpad=0.4,
    fontsize=secondary_font*0.75
)

ax.tick_params(axis='both', labelsize=secondary_font*0.85)



#fig.savefig('d18O_trens.pdf')
#fig.savefig('d18O_trens.png')

In [ ]:
df_sorted_indexed = df_sorted.set_index('age_dt')
primary_font = 15
secondary_font = 12

fig, ax = plt.subplots(1, figsize=(12,6))

ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d18o'], ".", color='#3492eb', alpha=0.7, label='main sampling' )
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d18o_sg_trend'],  color='#0D0454', lw=2.5, label='filtered' )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['calendar_year'], fred['d18o_SMOW'], ".", color='#40E0D0', alpha=0.7, label='prelimanary sampling'  )
#ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

#plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
plt.axvline(2007+(31+29+31)/365, color='#EE27F5', label='M8.1 EQ 2007 Apr')



#date1 = '1936-04-19'
date1 = '1944-08-01'
#date2 = '1944-06'
date2 = '1947-12'
date2a = '1953-02'
#date3 = '1966-02'
date3 = '1970-10'
date4 = '1979-02'
date5 = '1992-06'
date6 = '2007-01'
date7 = '2010-01'

                   ###########  1
x, y, p= line_fit(df_sorted_indexed.loc[:date1,'age_final'].values, 
               df_sorted_indexed.loc[:date1,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean(), y.mean()-0.2, f'{p[0]:.3f}', color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  2
x, y, p= line_fit(df_sorted_indexed.loc[date1:date2,'age_final'].values, 
               df_sorted_indexed.loc[date1:date2,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '-', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


x, y, p= line_fit(df_sorted_indexed.loc[date2:date2a,'age_final'].values, 
               df_sorted_indexed.loc[date2:date2a,'d18o'].values)
ax.plot(x, y, linestyle='dashed', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '--', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-2, y.mean()-0.2, f'{p[0]:.3f}', color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  3
x, y, p= line_fit(df_sorted_indexed.loc[date2a:date3,'age_final'].values, 
               df_sorted_indexed.loc[date2a:date3,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '-', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  4
x, y, p= line_fit(df_sorted_indexed.loc[date3:date4,'age_final'].values, 
               df_sorted_indexed.loc[date3:date4,'d18o'].values)
ax.plot(x, y, linestyle='dashed', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '--', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  5
x, y, p= line_fit(df_sorted_indexed.loc[date4:date5,'age_final'].values, 
               df_sorted_indexed.loc[date4:date5,'d18o'].values)
ax.plot(x, y, linestyle='dashed', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '--', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()-0.2, f'{p[0]:.3f}', color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  6
x, y, p= line_fit(df_sorted_indexed.loc[date5:date6,'age_final'].values, 
               df_sorted_indexed.loc[date5:date6,'d18o'].values)
ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+0.5, y.mean()+0.2, f'{p[0]:.3f}', color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  7
x, y, p= line_fit(fred_indexed.loc[:date6,'calendar_year'].values, 
               fred_indexed.loc[:date6,'d18o_SMOW'].values)
ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-1, y.mean()-0.4, f'{p[0]:.3f}', color='#178C81', bbox={'edgecolor':'#178C81' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})



                   ###########  8
x, y, p= line_fit(fred_indexed.loc[date6:date7,'calendar_year'].values, 
               fred_indexed.loc[date6:date7,'d18o_SMOW'].values)
ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()+0.2, f'{p[0]:.3f}', color='#0A3D38', bbox={'edgecolor':'#0A3D38' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

for d in [date1, date2, date2a, date3, date4, date5, date6, date7]:
    plt.axvline(x=datetime_to_decimal_year(pd.to_datetime( d)),
               #ymin=-3.5, ymax=-1,linestyle='dashed',color='gray', alpha=0.6)
                linestyle='dashed',color='gray', alpha=0.6)
ax.set_ylabel(r'$\delta^{18}$O [PDB]', fontdict={'fontsize': 15})
ax.set_xlabel('Year', fontdict={'fontsize': 15})

#ax.set_ylim(-3, 0.5)
ax.set_xlim(1936, 2012)

#ax.grid()
#ax.legend(loc='lower left')
#ax.legend(bbox_to_anchor=(1, 1))
ax.legend(
    #loc=(0.56, 0.6),
    loc='lower left',
    #bbox_to_anchor=(0.5, -0.15),  # Centered below the plot
    #ncol=2,                       # 3 columns
    frameon=True,
    facecolor='white',
    #columnspacing=1.5,
    #handletextpad=0.4,
    fontsize=secondary_font*0.75
)

ax.tick_params(axis='both', labelsize=secondary_font*0.85)



#fig.savefig('d18O_trens.pdf')
#fig.savefig('d18O_trens.png')

In [ ]:
df_sorted_indexed = df_sorted.set_index('age_dt')
primary_font = 15
secondary_font = 13

fig, axes = plt.subplots(2, figsize=(12,10))

ax= axes[0]
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d13c'], ".", color='lightgreen', alpha=0.7, label='main sampling' )
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d13c_sg_trend'],  color='darkgreen', lw=2.5, label='filtered' )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['calendar_year'], fred['d13c'], ".", color='#40E0D0', alpha=0.7, label='prelimanary sampling'  )
#ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

#plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
ax.axvline(2007+(31+29+31)/365, color='#EE27F5', label='M8.1 EQ 2007 Apr')



#date1 = '1936-04-19'
date1 = '1944-08-01'
#date2 = '1944-06'
date2 = '1950-11'
date2a = '1959-01'
#date3 = '1966-02'
date3 = '1970-10'
date4 = '1979-02'
date5 = '1991-01'
date6 = '2007-01'
date7 = '2010-01'

                   ###########  1
x, y, p= line_fit(df_sorted_indexed.loc[:date1,'age_final'].values, 
               df_sorted_indexed.loc[:date1,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean(), y.mean()-0.2, f'{p[0]:.3f}',fontsize=secondary_font, color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  2
x, y, p= line_fit(df_sorted_indexed.loc[date1:date2,'age_final'].values, 
               df_sorted_indexed.loc[date1:date2,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '-', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+1, y.mean(), f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


x, y, p= line_fit(df_sorted_indexed.loc[date2:date2a,'age_final'].values, 
               df_sorted_indexed.loc[date2:date2a,'d13c'].values)
ax.plot(x, y, linestyle='dashed', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '--', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-2, y.mean()-0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  3
x, y, p= line_fit(df_sorted_indexed.loc[date2a:date3,'age_final'].values, 
               df_sorted_indexed.loc[date2a:date3,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '-', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', fontsize=secondary_font, color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  4
x, y, p= line_fit(df_sorted_indexed.loc[date3:date4,'age_final'].values, 
               df_sorted_indexed.loc[date3:date4,'d13c'].values)
ax.plot(x, y, linestyle='dashed', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '--', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  5
x, y, p= line_fit(df_sorted_indexed.loc[date4:date5,'age_final'].values, 
               df_sorted_indexed.loc[date4:date5,'d13c'].values)
ax.plot(x, y, linestyle='dashed', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '--', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()-0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  6
x, y, p= line_fit(df_sorted_indexed.loc[date5:date6,'age_final'].values, 
               df_sorted_indexed.loc[date5:date6,'d13c'].values)
ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+0.5, y.mean()+0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  7
x, y, p= line_fit(fred_indexed.loc[:date6,'calendar_year'].values, 
               fred_indexed.loc[:date6,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-1, y.mean()-0.4, f'{p[0]:.3f}', fontsize=secondary_font, color='#178C81', bbox={'edgecolor':'#178C81' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})



                   ###########  8
x, y, p= line_fit(fred_indexed.loc[date6:date7,'calendar_year'].values, 
               fred_indexed.loc[date6:date7,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()+0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#0A3D38', bbox={'edgecolor':'#0A3D38' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

for d in [date1, date2, date2a, date3, date4, date5, date6, date7]:
    ax.axvline(x=datetime_to_decimal_year(pd.to_datetime( d)),
                linestyle='dashed',color='gray', alpha=0.6)
ax.set_ylabel(r'$\delta^{13}$C [PDB]', fontdict={'fontsize': 15})
ax.set_xlabel('Year', fontdict={'fontsize': 15})

ax.set_ylim(-3, 0.5)
ax.set_xlim(1936, 2012)

#ax.grid()
#ax.legend(loc='lower left')
#ax.legend(bbox_to_anchor=(1, 1))
ax.legend(
    #loc=(0.56, 0.6),
    loc='lower left',
    #bbox_to_anchor=(0.5, -0.15),  # Centered below the plot
    #ncol=2,                       # 3 columns
    frameon=True,
    facecolor='white',
    #columnspacing=1.5,
    #handletextpad=0.4,
    fontsize=secondary_font
)

ax.tick_params(axis='both', labelsize=secondary_font)




ax = axes[1]
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d18o'], ".", color='#3492eb', alpha=0.7, label='main sampling' )
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d18o_sg_trend'],  color='#0D0454', lw=2.5, label='filtered' )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['calendar_year'], fred['d18o_SMOW'], ".", color='#40E0D0', alpha=0.7, label='prelimanary sampling'  )
#ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

#plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
plt.axvline(2007+(31+29+31)/365, color='#EE27F5', label='M8.1 EQ 2007 Apr')



#date1 = '1936-04-19'
date1 = '1944-08-01'
#date2 = '1944-06'
date2 = '1947-12'
date2a = '1953-02'
#date3 = '1966-02'
date3 = '1970-10'
date4 = '1979-02'
date5 = '1991-01'
date6 = '2007-01'
date7 = '2010-01'

                   ###########  1
x, y, p= line_fit(df_sorted_indexed.loc[:date1,'age_final'].values, 
               df_sorted_indexed.loc[:date1,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.2, y.mean()-0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  2
x, y, p= line_fit(df_sorted_indexed.loc[date1:date2,'age_final'].values, 
               df_sorted_indexed.loc[date1:date2,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '-', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+1, y.mean()+0.05, f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


x, y, p= line_fit(df_sorted_indexed.loc[date2:date2a,'age_final'].values, 
               df_sorted_indexed.loc[date2:date2a,'d18o'].values)
ax.plot(x, y, linestyle='dashed', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '--', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()-0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  3
x, y, p= line_fit(df_sorted_indexed.loc[date2a:date3,'age_final'].values, 
               df_sorted_indexed.loc[date2a:date3,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '-', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean()+0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  4
x, y, p= line_fit(df_sorted_indexed.loc[date3:date5,'age_final'].values, 
               df_sorted_indexed.loc[date3:date5,'d18o'].values)
ax.plot(x, y, linestyle='dashed', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '--', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean()+0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  6
x, y, p= line_fit(df_sorted_indexed.loc[date5:date6,'age_final'].values, 
               df_sorted_indexed.loc[date5:date6,'d18o'].values)
ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+0.25, y.mean()+0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  7
x, y, p= line_fit(fred_indexed.loc[:date6,'calendar_year'].values, 
               fred_indexed.loc[:date6,'d18o_SMOW'].values)
ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-1, y.mean()-0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#178C81', bbox={'edgecolor':'#178C81' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})



                   ###########  8
x, y, p= line_fit(fred_indexed.loc[date6:,'calendar_year'].values, 
               fred_indexed.loc[date6:,'d18o_SMOW'].values)
ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-1.8, y.mean()+0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#0A3D38', bbox={'edgecolor':'#0A3D38' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

for d in [date1, date2, date2a, date3, date5, date6]:
    plt.axvline(x=datetime_to_decimal_year(pd.to_datetime( d)),
               #ymin=-3.5, ymax=-1,linestyle='dashed',color='gray', alpha=0.6)
                linestyle='dashed',color='gray', alpha=0.6)
ax.set_ylabel(r'$\delta^{18}$O [SMOW]', fontdict={'fontsize': 15})
ax.set_xlabel('Year', fontdict={'fontsize': 15})

#ax.set_ylim(-3, 0.5)
ax.set_xlim(1936, 2012)

#ax.grid()
#ax.legend(loc='lower left')
#ax.legend(bbox_to_anchor=(1, 1))
ax.legend(
    #loc=(0.56, 0.6),
    loc='lower left',
    #bbox_to_anchor=(0.5, -0.15),  # Centered below the plot
    #ncol=2,                       # 3 columns
    frameon=True,
    facecolor='white',
    #columnspacing=1.5,
    #handletextpad=0.4,
    fontsize=secondary_font
)

ax.tick_params(axis='both', labelsize=secondary_font)

axes[0].set_title(r'Linear Trends of $\delta^{13}$C & $\delta^{18}$O', fontdict={'fontsize': 20})

#fig.savefig('d13c-d18O_trens.pdf')
#fig.savefig('d13c-d18O_trens.png')

In [ ]:
df_sorted_indexed = df_sorted.set_index('age_dt')
primary_font = 15
secondary_font = 13
color_vline = '#F3B939'

from matplotlib.patches import ConnectionPatch

# Split the morphology panel into two rows in the gridspec
fig, axes = plt.subplots(4, figsize=(12, 12),
                         gridspec_kw={'height_ratios': [0.2, 1, 2, 2],
                                      'hspace': 0})

ax= axes[2]
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d13c'], ".", color='lightgreen', alpha=0.7, label='main sampling' )
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d13c_sg_trend'],  color='darkgreen', lw=2.5, label='filtered' )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['calendar_year'], fred['d13c'], ".", color='#40E0D0', alpha=0.7, label='prelimanary sampling'  )
#ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

#plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
ax.axvline(2007+(31+29+31)/365, color='#B30BBF', label='M8.1 EQ 2007 Apr')



#date1 = '1936-04-19'
date1 = '1944-08-01'
#date2 = '1944-06'
date2 = '1950-11'
date2a = '1959-01'
#date3 = '1966-02'
date3 = '1970-10'
date4 = '1979-02'
date5 = '1991-01'
date6 = '2007-01'
date7 = '2010-01'

                   ###########  1
x, y, p= line_fit(df_sorted_indexed.loc[:date1,'age_final'].values, 
               df_sorted_indexed.loc[:date1,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean(), y.mean()-0.2, f'{p[0]:.3f}',fontsize=secondary_font, color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  2
x, y, p= line_fit(df_sorted_indexed.loc[date1:date2,'age_final'].values, 
               df_sorted_indexed.loc[date1:date2,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '-', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+1, y.mean(), f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


x, y, p= line_fit(df_sorted_indexed.loc[date2:date2a,'age_final'].values, 
               df_sorted_indexed.loc[date2:date2a,'d13c'].values)
ax.plot(x, y, linestyle='dashed', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '--', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-2, y.mean()-0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  3
x, y, p= line_fit(df_sorted_indexed.loc[date2a:date3,'age_final'].values, 
               df_sorted_indexed.loc[date2a:date3,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '-', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', fontsize=secondary_font, color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  4
x, y, p= line_fit(df_sorted_indexed.loc[date3:date4,'age_final'].values, 
               df_sorted_indexed.loc[date3:date4,'d13c'].values)
ax.plot(x, y, linestyle='dashed', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '--', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  5
x, y, p= line_fit(df_sorted_indexed.loc[date4:date5,'age_final'].values, 
               df_sorted_indexed.loc[date4:date5,'d13c'].values)
ax.plot(x, y, linestyle='dashed', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '--', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()-0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  6
x, y, p= line_fit(df_sorted_indexed.loc[date5:date6,'age_final'].values, 
               df_sorted_indexed.loc[date5:date6,'d13c'].values)
ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+0.5, y.mean()+0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  7
x, y, p= line_fit(fred_indexed.loc[:date6,'calendar_year'].values, 
               fred_indexed.loc[:date6,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-1, y.mean()-0.4, f'{p[0]:.3f}', fontsize=secondary_font, color='#178C81', bbox={'edgecolor':'#178C81' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})



                   ###########  8
x, y, p= line_fit(fred_indexed.loc[date6:date7,'calendar_year'].values, 
               fred_indexed.loc[date6:date7,'d13c'].values)
ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()+0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#0A3D38', bbox={'edgecolor':'#0A3D38' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

for d in [date1, date2, date2a, date3, date4, date5, date6, date7]:
    ax.axvline(x=datetime_to_decimal_year(pd.to_datetime( d)),
                linestyle='dashed',color=color_vline, alpha=0.6)
ax.set_ylabel(r'$\delta^{13}$C [PDB]', fontdict={'fontsize': 15})
ax.set_xlabel('Year', fontdict={'fontsize': 15})

ax.set_ylim(-3, 0.5)
ax.set_xlim(1936, 2012)

#ax.grid()
#ax.legend(loc='lower left')
#ax.legend(bbox_to_anchor=(1, 1))
ax.legend(
    #loc=(0.56, 0.6),
    loc='lower left',
    #bbox_to_anchor=(0.5, -0.15),  # Centered below the plot
    #ncol=2,                       # 3 columns
    frameon=True,
    facecolor='white',
    #columnspacing=1.5,
    #handletextpad=0.4,
    fontsize=secondary_font
)

ax.tick_params(axis='both', labelsize=secondary_font)
ax.set_yticks(np.arange(-3,0.5,0.5))




ax = axes[3]
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d18o'], ".", color='#3492eb', alpha=0.7, label='main sampling' )
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d18o_sg_trend'],  color='#0D0454', lw=2.5, label='filtered' )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['calendar_year'], fred['d18o_SMOW'], ".", color='#40E0D0', alpha=0.7, label='prelimanary sampling'  )
#ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

#plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
plt.axvline(2007+(31+29+31)/365, color='#B30BBF', label='M8.1 EQ 2007 Apr')



#date1 = '1936-04-19'
date1 = '1944-08-01'
#date2 = '1944-06'
date2 = '1947-12'
date2a = '1953-02'
#date3 = '1966-02'
date3 = '1970-10'
date4 = '1979-02'
date5 = '1991-01'
date6 = '2007-01'
date7 = '2010-01'

                   ###########  1
x, y, p= line_fit(df_sorted_indexed.loc[:date1,'age_final'].values, 
               df_sorted_indexed.loc[:date1,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.2, y.mean()-0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  2
x, y, p= line_fit(df_sorted_indexed.loc[date1:date2,'age_final'].values, 
               df_sorted_indexed.loc[date1:date2,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '-', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+1, y.mean()+0.05, f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


x, y, p= line_fit(df_sorted_indexed.loc[date2:date2a,'age_final'].values, 
               df_sorted_indexed.loc[date2:date2a,'d18o'].values)
ax.plot(x, y, linestyle='dashed', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '--', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()-0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  3
x, y, p= line_fit(df_sorted_indexed.loc[date2a:date3,'age_final'].values, 
               df_sorted_indexed.loc[date2a:date3,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '-', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean()+0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  4
x, y, p= line_fit(df_sorted_indexed.loc[date3:date5,'age_final'].values, 
               df_sorted_indexed.loc[date3:date5,'d18o'].values)
ax.plot(x, y, linestyle='dashed', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '--', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean()+0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  6
x, y, p= line_fit(df_sorted_indexed.loc[date5:date6,'age_final'].values, 
               df_sorted_indexed.loc[date5:date6,'d18o'].values)
ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+0.25, y.mean()+0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  7
x, y, p= line_fit(fred_indexed.loc[:date6,'calendar_year'].values, 
               fred_indexed.loc[:date6,'d18o_SMOW'].values)
ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-1, y.mean()-0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#178C81', bbox={'edgecolor':'#178C81' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})



                   ###########  8
x, y, p= line_fit(fred_indexed.loc[date6:,'calendar_year'].values, 
               fred_indexed.loc[date6:,'d18o_SMOW'].values)
ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-1.8, y.mean()+0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#0A3D38', bbox={'edgecolor':'#0A3D38' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

for d in [date1, date2, date2a, date3, date5, date6]:
    ax.axvline(x=datetime_to_decimal_year(pd.to_datetime( d)),
               #ymin=-3.5, ymax=-1,linestyle='dashed',color='gray', alpha=0.6)
                linestyle='dashed',color=color_vline, alpha=0.6)
ax.set_ylabel(r'$\delta^{18}$O [SMOW]', fontdict={'fontsize': 15})
ax.set_xlabel('Year', fontdict={'fontsize': 15})

#ax.set_ylim(-3, 0.5)
ax.set_xlim(1936, 2012)

#ax.grid()
#ax.legend(loc='lower left')
#ax.legend(bbox_to_anchor=(1, 1))
ax.legend(
    #loc=(0.56, 0.6),
    loc='lower left',
    #bbox_to_anchor=(0.5, -0.15),  # Centered below the plot
    #ncol=2,                       # 3 columns
    frameon=True,
    facecolor='white',
    #columnspacing=1.5,
    #handletextpad=0.4,
    fontsize=secondary_font
)

ax.tick_params(axis='both', labelsize=secondary_font)

#date1 = '1936-04-19'
date1 = '1944-08-01'
#date2 = '1944-06'
date2 = '1950-11'
date2a = '1959-01'
#date3 = '1966-02'
date3 = '1970-10'
date4 = '1979-02'
date5 = '1991-01'
date6 = '2007-01'
date7 = '2010-01'


# ── Morphology: broken y-axis ───────────────────────────────────────────────
ax_top = axes[0]   # upper panel: spike  −90 → −35
ax_bot = axes[1]   # lower panel: body  −130 → −90

x_m = morpho_memo['age_final'].values
y_m = morpho_memo['y_true'].values

for ax in (ax_top, ax_bot):
    ax.plot(x_m, y_m, color='#BF290B', lw=2.5, alpha=0.85, label='Reversed coral'+'\n'+'morphology')
    ax.set_xlim(1936, 2012)
    ax.tick_params(axis='both', labelsize=secondary_font)
    for d in [date1, date2,  date3, date4, date5, date6]:
        
        ax.axvline(x=datetime_to_decimal_year(pd.to_datetime(d)),
                   linestyle='dashed', color=color_vline, alpha=0.6)
    ax.axvline(2007 + (31+29+31)/365, color='#B30BBF', label='M8.1 EQ 2007 Apr')

# Upper panel: coseismic spike only
ax_top.set_ylim(-50, -35)
ax_top.spines['bottom'].set_visible(False)
ax_top.tick_params(bottom=False, labelbottom=False)
ax_top.tick_params(axis='y', labelsize=secondary_font,
                   left=False, labelleft=False,
                   right=True, labelright=True)
ax_top.yaxis.tick_right()
ax_top.xaxis.tick_top()
ax_top.tick_params(axis='x', top=True, labeltop=True)

# Lower panel: main body
ax_bot.set_ylim(-130, -90)
ax_bot.spines['top'].set_visible(False)
ax_bot.set_xlabel('Year', fontdict={'fontsize': secondary_font})
ax_bot.tick_params(axis='y', labelsize=secondary_font,
                   left=True, labelleft=True,
                   right=False, labelright=False)
ax_bot.set_ylabel('Realtive Water'+'\n'+'Depth [cm]', fontdict={'fontsize': secondary_font})
ax_bot.yaxis.set_label_position('left')
ax_bot.legend(loc='lower left', fontsize=secondary_font)

# ── Diagonal break marks ─────────────────────────────────────────────────────
kw = dict(color='k', clip_on=False, lw=1.2, transform=ax_top.transAxes)
ax_top.plot([-0.005,  0.005], [-0.05,  0.05], **kw)
ax_top.plot([ 0.995,  1.005], [-0.05,  0.05], **kw)

kw['transform'] = ax_bot.transAxes
ax_bot.plot([-0.005,  0.005], [0.95, 1.05], **kw)
ax_bot.plot([ 0.995,  1.005], [0.95, 1.05], **kw)

ax_top.grid(axis='x', alpha=0.6)
ax_bot.grid(axis='x', alpha=0.6)
axes[2].grid(axis='x', alpha=0.6)
axes[3].grid(axis='x', alpha=0.6)

axes[0].set_title(r'Linear Trends of $\delta^{13}$C & $\delta^{18}$O', fontdict={'fontsize': 20}, pad=50)

fig.tight_layout()
#fig.savefig('d13c-d18O_trends_morpho.pdf')
#fig.savefig('d13c-d18O_trends_morpho.png')

In [ ]:
df_sorted_indexed = df_sorted.set_index('age_dt')
primary_font = 15
secondary_font = 13
color_vline = '#F3B939'

from matplotlib.patches import ConnectionPatch

# Split the morphology panel into two rows in the gridspec
fig, axes = plt.subplots(4, figsize=(12, 12),
                         gridspec_kw={'height_ratios': [0.2, 1, 2, 2],
                                      'hspace': 0})

ax= axes[2]
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d13c_suess'], ".", color='lightgreen', alpha=0.7, label='main sampling' )
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d13c_suess_sg_trend'],  color='darkgreen', lw=2.5, label='filtered' )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['calendar_year'], fred['d13c_suess'], ".", color='#40E0D0', alpha=0.7, label='prelimanary sampling'  )
#ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

#plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
ax.axvline(2007+(31+29+31)/365, color='#B30BBF', label='M8.1 EQ 2007 Apr')



#date1 = '1936-04-19'
date1 = '1944-08-01'
#date2 = '1944-06'
date2 = '1950-11'
date2a = '1959-01'
#date3 = '1966-02'
date3 = '1970-10'
date4 = '1979-02'
date5 = '1991-01'
date6 = '2007-01'
date7 = '2010-01'

                   ###########  1
x, y, p= line_fit(df_sorted_indexed.loc[:date1,'age_final'].values, 
               df_sorted_indexed.loc[:date1,'d13c_suess'].values)
ax.plot(x, y, linestyle='solid', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean(), y.mean()-0.2, f'{p[0]:.3f}',fontsize=secondary_font, color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  2
x, y, p= line_fit(df_sorted_indexed.loc[date1:date2,'age_final'].values, 
               df_sorted_indexed.loc[date1:date2,'d13c_suess'].values)
ax.plot(x, y, linestyle='solid', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '-', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+1, y.mean(), f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


x, y, p= line_fit(df_sorted_indexed.loc[date2:date2a,'age_final'].values, 
               df_sorted_indexed.loc[date2:date2a,'d13c_suess'].values)
ax.plot(x, y, linestyle='dashed', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '--', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-2, y.mean()-0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  3
x, y, p= line_fit(df_sorted_indexed.loc[date2a:date3,'age_final'].values, 
               df_sorted_indexed.loc[date2a:date3,'d13c_suess'].values)
ax.plot(x, y, linestyle='solid', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '-', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', fontsize=secondary_font, color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  4
x, y, p= line_fit(df_sorted_indexed.loc[date3:date4,'age_final'].values, 
               df_sorted_indexed.loc[date3:date4,'d13c_suess'].values)
ax.plot(x, y, linestyle='dashed', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '--', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean(), f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  5
x, y, p= line_fit(df_sorted_indexed.loc[date4:date5,'age_final'].values, 
               df_sorted_indexed.loc[date4:date5,'d13c_suess'].values)
ax.plot(x, y, linestyle='dashed', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '--', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()-0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  6
x, y, p= line_fit(df_sorted_indexed.loc[date5:date6,'age_final'].values, 
               df_sorted_indexed.loc[date5:date6,'d13c_suess'].values)
ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+0.5, y.mean()+0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  7
x, y, p= line_fit(fred_indexed.loc[:date6,'calendar_year'].values, 
               fred_indexed.loc[:date6,'d13c_suess'].values)
ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-1, y.mean()-0.4, f'{p[0]:.3f}', fontsize=secondary_font, color='#178C81', bbox={'edgecolor':'#178C81' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})



                   ###########  8
x, y, p= line_fit(fred_indexed.loc[date6:date7,'calendar_year'].values, 
               fred_indexed.loc[date6:date7,'d13c_suess'].values)
ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()+0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#0A3D38', bbox={'edgecolor':'#0A3D38' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

for d in [date1, date2, date2a, date3, date4, date5, date6, date7]:
    ax.axvline(x=datetime_to_decimal_year(pd.to_datetime( d)),
                linestyle='dashed',color=color_vline, alpha=0.6)
ax.set_ylabel(r'$\delta^{13}$C [PDB]', fontdict={'fontsize': 15})
ax.set_xlabel('Year', fontdict={'fontsize': 15})

ax.set_ylim(-1.5, 0.5)
ax.set_xlim(1936, 2012)
ax.set_yticks(np.arange(-1.5,0.5,0.25))

#ax.grid()
#ax.legend(loc='lower left')
#ax.legend(bbox_to_anchor=(1, 1))
ax.legend(
    #loc=(0.56, 0.6),
    loc='upper left',
    #bbox_to_anchor=(0.5, -0.15),  # Centered below the plot
    #ncol=2,                       # 3 columns
    frameon=True,
    facecolor='white',
    #columnspacing=1.5,
    #handletextpad=0.4,
    fontsize=secondary_font
)

ax.tick_params(axis='both', labelsize=secondary_font)
ax.tick_params(axis='y', right=True, labelright=True, labelsize=secondary_font)


####################################################################################################################
####################################          d18O Plot                #############################################
####################################################################################################################


ax = axes[3]
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d18o'], ".", color='#3492eb', alpha=0.7, label='main sampling' )
ax.plot(df_sorted_indexed['age_final'], df_sorted_indexed['d18o_sg_trend'],  color='#0D0454', lw=2.5, label='filtered' )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['calendar_year'], fred['d18o_SMOW'], ".", color='#40E0D0', alpha=0.7, label='prelimanary sampling'  )
#ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

#plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
plt.axvline(2007+(31+29+31)/365, color='#B30BBF', label='M8.1 EQ 2007 Apr')



#date1 = '1936-04-19'
date1 = '1944-08-01'
#date2 = '1944-06'
date2 = '1947-12'
date2a = '1953-02'
#date3 = '1966-02'
date3 = '1970-10'
date4 = '1979-02'
date5 = '1991-01'
date6 = '2007-01'
date7 = '2010-01'

                   ###########  1
x, y, p= line_fit(df_sorted_indexed.loc[:date1,'age_final'].values, 
               df_sorted_indexed.loc[:date1,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.2, y.mean()-0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

                   ###########  2
x, y, p= line_fit(df_sorted_indexed.loc[date1:date2,'age_final'].values, 
               df_sorted_indexed.loc[date1:date2,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '-', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+1, y.mean()+0.05, f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


x, y, p= line_fit(df_sorted_indexed.loc[date2:date2a,'age_final'].values, 
               df_sorted_indexed.loc[date2:date2a,'d18o'].values)
ax.plot(x, y, linestyle='dashed', color='#CF1352', linewidth=2.5)
#ax.plot(x, y, '--', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-0.5, y.mean()-0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#CF1352', bbox={'edgecolor':'#CF1352' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  3
x, y, p= line_fit(df_sorted_indexed.loc[date2a:date3,'age_final'].values, 
               df_sorted_indexed.loc[date2a:date3,'d18o'].values)
ax.plot(x, y, linestyle='solid', color='#FF9724', linewidth=2.5)
#ax.plot(x, y, '-', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean()+0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#FF9724', bbox={'edgecolor':'#FF9724' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  4
x, y, p= line_fit(df_sorted_indexed.loc[date3:date5,'age_final'].values, 
               df_sorted_indexed.loc[date3:date5,'d18o'].values)
ax.plot(x, y, linestyle='dashed', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, '--', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+2, y.mean()+0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  6
x, y, p= line_fit(df_sorted_indexed.loc[date5:date6,'age_final'].values, 
               df_sorted_indexed.loc[date5:date6,'d18o'].values)
ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5)
#ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()+0.25, y.mean()+0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#1316CF', bbox={'edgecolor':'#1316CF' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})


                   ###########  7
x, y, p= line_fit(fred_indexed.loc[:date6,'calendar_year'].values, 
               fred_indexed.loc[:date6,'d18o_SMOW'].values)
ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#178C81', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-1, y.mean()-0.2, f'{p[0]:.3f}', fontsize=secondary_font, color='#178C81', bbox={'edgecolor':'#178C81' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})



                   ###########  8
x, y, p= line_fit(fred_indexed.loc[date6:,'calendar_year'].values, 
               fred_indexed.loc[date6:,'d18o_SMOW'].values)
ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5)
#ax.plot(x, y, linestyle='solid', color='#0A3D38', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{13}$C ‰ / a')
ax.text(x.mean()-1.8, y.mean()+0.1, f'{p[0]:.3f}', fontsize=secondary_font, color='#0A3D38', bbox={'edgecolor':'#0A3D38' ,'facecolor': 'white', 'alpha': 1, 'pad': 1})

for d in [date1, date2, date2a, date3, date5, date6]:
    ax.axvline(x=datetime_to_decimal_year(pd.to_datetime( d)),
               #ymin=-3.5, ymax=-1,linestyle='dashed',color='gray', alpha=0.6)
                linestyle='dashed',color=color_vline, alpha=0.6)
ax.set_ylabel(r'$\delta^{18}$O [SMOW]', fontdict={'fontsize': 15})
ax.set_xlabel('Year', fontdict={'fontsize': 15})

#ax.set_ylim(-3, 0.5)
ax.set_xlim(1936, 2012)

#ax.grid()
#ax.legend(loc='lower left')
#ax.legend(bbox_to_anchor=(1, 1))
ax.legend(
    #loc=(0.56, 0.6),
    loc='lower left',
    #bbox_to_anchor=(0.5, -0.15),  # Centered below the plot
    #ncol=2,                       # 3 columns
    frameon=True,
    facecolor='white',
    #columnspacing=1.5,
    #handletextpad=0.4,
    fontsize=secondary_font
)

ax.tick_params(axis='both', labelsize=secondary_font)
ax.tick_params(axis='y', right=True, labelright=True, labelsize=secondary_font)

#date1 = '1936-04-19'
date1 = '1944-08-01'
#date2 = '1944-06'
date2 = '1950-11'
date2a = '1959-01'
#date3 = '1966-02'
date3 = '1970-10'
date4 = '1979-02'
date5 = '1991-01'
date6 = '2007-01'
date7 = '2010-01'


# ── Morphology: broken y-axis ───────────────────────────────────────────────
ax_top = axes[0]   # upper panel: spike  −90 → −35
ax_bot = axes[1]   # lower panel: body  −130 → −90

x_m = morpho_memo['age_final'].values
y_m = morpho_memo['y_true'].values

for ax in (ax_top, ax_bot):
    ax.plot(x_m, y_m, color='#BF290B', lw=2.5, alpha=0.85, label='Reversed coral'+'\n'+'morphology')
    ax.set_xlim(1936, 2012)
    ax.tick_params(axis='both', labelsize=secondary_font)
    for d in [date1, date2,  date3, date4, date5, date6]:
        
        ax.axvline(x=datetime_to_decimal_year(pd.to_datetime(d)),
                   linestyle='dashed', color=color_vline, alpha=0.6)
    ax.axvline(2007 + (31+29+31)/365, color='#B30BBF', label='M8.1 EQ 2007 Apr')

# Upper panel: coseismic spike only
ax_top.set_ylim(-50, -35)
ax_top.spines['bottom'].set_visible(False)
ax_top.tick_params(bottom=False, labelbottom=False)
ax_top.tick_params(axis='y', labelsize=secondary_font,
                   left=False, labelleft=False,
                   right=True, labelright=True)
ax_top.yaxis.tick_right()
ax_top.xaxis.tick_top()
ax_top.tick_params(axis='x', top=True, labeltop=True)

# Lower panel: main body
ax_bot.set_ylim(-130, -90)
ax_bot.spines['top'].set_visible(False)
ax_bot.set_xlabel('Year', fontdict={'fontsize': secondary_font})
ax_bot.tick_params(axis='y', labelsize=secondary_font,
                   left=True, labelleft=True,
                   right=False, labelright=False)
ax_bot.set_ylabel('Realtive Water'+'\n'+'Depth [cm]', fontdict={'fontsize': secondary_font})
ax_bot.yaxis.set_label_position('left')
ax_bot.legend(loc='lower left', fontsize=secondary_font)

# ── Diagonal break marks ─────────────────────────────────────────────────────
kw = dict(color='k', clip_on=False, lw=1.2, transform=ax_top.transAxes)
ax_top.plot([-0.005,  0.005], [-0.05,  0.05], **kw)
ax_top.plot([ 0.995,  1.005], [-0.05,  0.05], **kw)

kw['transform'] = ax_bot.transAxes
ax_bot.plot([-0.005,  0.005], [0.95, 1.05], **kw)
ax_bot.plot([ 0.995,  1.005], [0.95, 1.05], **kw)

ax_top.grid(axis='x', alpha=0.6)
ax_bot.grid(axis='x', alpha=0.6)
axes[2].grid(axis='x', alpha=0.6)
axes[3].grid(axis='x', alpha=0.6)

axes[0].set_title(r'Linear Trends of $\delta^{13}$C & $\delta^{18}$O', fontdict={'fontsize': 20}, pad=50)

fig.tight_layout()
fig.savefig('d13c-d18O_trends_morpho_suess.pdf')
fig.savefig('d13c-d18O_trends_morpho_suess.png')

In [ ]:
print(morpho_memo['age_final'].dtype)
print(morpho_memo['age_final'].head(10))
print(morpho_memo['age_final'].min(), morpho_memo['age_final'].max())
print(morpho_memo['y_true'].min(), morpho_memo['y_true'].max())

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="markers", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="lines", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_resampled['age_resampled'], y=df_resampled['d13c'] , mode="markers", line_color='#C20A20',text=df_sorted['Name'], name='raw'), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

#fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
fig.update_layout(width=1000, height=500, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')

# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c_no_tectonic'] , mode="lines", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="lines", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)

#fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_resampled['age_resampled'], y=df_resampled['d13c_no_tectonic'] , mode="markers", line_color='#C20A20',text=df_sorted['Name'], name='raw'), secondary_y=False)


fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

#fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
fig.update_layout(width=1000, height=500, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')

# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:


fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d18o'] , mode="markers", line_color='#3492eb',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d18o'] , mode="lines", line_color='#3492eb',text=df_sorted['Name'], name='raw'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d18o_ma'] , mode="lines", line_color='#134b80', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d18o_ma2'] , mode="lines", line_color='#0D0454', name='3 year average'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_resampled['age_resampled'], y=df_resampled['d18o'] , mode="markers", line_color='#C20A20',text=df_sorted['Name'], name='raw'), secondary_y=False)

#fig.add_trace(go.Scatter(x=aligned['age_absolute'], y=aligned['d18o_no_tec_sg']+25.75 , mode="lines", line_color='green',text=df_sorted['Name'], name='aligned_d18o'), secondary_y=True)



#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=2), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:
df_resampled['d18o']

In [ ]:
# Ensure df_resampled is sorted by date
df_resampled_sorted = df_resampled[['d13c', 'd13c_no_tectonic', 'd13c_no_tec_sg','d18o','d18o_pdb' ,'d18o_no_tectonic', 'd18o_no_tec_sg']].sort_index()


# Target dates to align with
#target_dates = pd.to_datetime(enso.loc[:685, 'Date'])
target_dates = pd.date_range('1950-01-31', '2007-01-31', freq='3ME')

# Interpolate to get values at those dates
aligned = df_resampled_sorted.reindex(df_resampled_sorted.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned['age_absolute'] = np.array([datetime_to_decimal_year(x) for x in pd.to_datetime(aligned.index)])


aligned_enso = enso['ENSO'].reindex(enso['ENSO'].index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned = aligned.join(aligned_enso.dropna())

aligned_tidegauge = tide.reindex(tide.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned = aligned.join(aligned_tidegauge['Sea Level'].dropna())

aligned_climate = climate.set_index('datetime')
aligned_climate = aligned_climate.reindex(aligned_climate.index.union(target_dates)).interpolate(method='time').loc[target_dates]

aligned = aligned.join(aligned_climate[['SST','SSS', 'synthetic_d18o']].dropna())

aligned.loc['2005-01-01':'2008-01-01']

In [ ]:
# ── ALIGNED: interpolate all external series to exact coral sample ages ─────
#
# WHY THE OLD APPROACH BROKE
# ---------------------------
# The previous code resampled coral to a 3ME grid, then joined ENSO / tide /
# climate via separate .reindex().interpolate().loc[target_dates] calls.
# Each join used a different intermediate index so after dropna() the surviving
# rows differed column by column — rolling().corr() then compared values at
# DIFFERENT calendar times, giving meaningless correlations.
#
# THE FIX
# -------
# Keep df_sorted at its exact irregular sample ages (decimal years).
# Map every external series onto those same ages with a single linear
# interpolator.  All columns share one index → rolling().corr() is correct.

from scipy.signal import savgol_filter
from scipy.interpolate import interp1d

def interp_to(x_ref, y_ref, target):
    """Piecewise-linear interpolation; returns NaN outside x_ref range."""
    x_ref  = np.asarray(x_ref,  float)
    y_ref  = np.asarray(y_ref,  float)
    target = np.asarray(target, float)
    out    = np.full(len(target), np.nan)
    for i, a in enumerate(target):
        idx = np.searchsorted(x_ref, a)
        if 0 < idx < len(x_ref):
            t      = (a - x_ref[idx-1]) / (x_ref[idx] - x_ref[idx-1])
            out[i] = y_ref[idx-1] + t * (y_ref[idx] - y_ref[idx-1])
    return out

# Use exact sample ages as the shared axis
ages = df_sorted['age_final'].values

# ── Savitzky-Golay tectonic detrend (matches read_data.py) ────────────────────
# Build a monthly grid, apply SG (73-month window, order 3), interpolate back.
def to_monthly(a): return (np.array(a) * 12).round() / 12

df_sorted_copy = df_sorted.copy()
df_sorted_copy['mb'] = to_monthly(ages)
coral_mon = df_sorted_copy.groupby('mb')[['d13c', 'd18o_pdb']].mean().reset_index()
coral_mon.rename(columns={'mb': 'age'}, inplace=True)

for col in ['d13c', 'd18o_pdb']:
    sg = savgol_filter(
        coral_mon[col].interpolate().ffill().bfill().values,
        window_length=73, polyorder=3, mode='interp'
    )
    fn = interp1d(coral_mon['age'].values, sg,
                  bounds_error=False, fill_value='extrapolate')
    df_sorted[f'{col}_sg_trend']  = fn(ages)
    df_sorted[f'{col}_no_tec_sg'] = df_sorted[col] - df_sorted[f'{col}_sg_trend']


# ── Climate ────────────────────────────────────────────────────────────────────
clim_s = climate.sort_values('age_absolute')
for col in ['SST', 'SSS', 'synthetic_d18o']:
    df_sorted[col] = interp_to(clim_s['age_absolute'].values, clim_s[col].values, ages)

# ── Tide gauge ────────────────────────────────────────────────────────────────
tide_s  = tide.dropna(subset=['Sea Level']).sort_values('Date')
sl_raw  = interp_to(tide_s['Date'].values, tide_s['Sea Level'].values, ages)
df_sorted['Sea Level'] = (sl_raw - np.nanmean(sl_raw)) / 1000.0   # mm → m, remove mean

# ── ENSO ──────────────────────────────────────────────────────────────────────
enso_dt  = pd.to_datetime(enso['Date'])
enso_age = (enso_dt.dt.year + (enso_dt.dt.month - 0.5) / 12).values
order    = np.argsort(enso_age)
df_sorted['ENSO'] = interp_to(enso_age[order], enso['ENSO'].values[order], ages)

# ── Build aligned: single float index, one row per coral sample ──────────────
aligned = df_sorted[[
    'age_final',
    'd13c', 'd18o', 'd18o_pdb',
    'd13c_sg_trend', 'd18o_pdb_sg_trend',        # ← added
    'd13c_no_tec_sg', 'd18o_pdb_no_tec_sg',
    'SST', 'SSS', 'synthetic_d18o',
    'Sea Level', 'ENSO'
]].rename(columns={'age_final': 'age'}).set_index('age').sort_index()

print(f"aligned: {len(aligned)} rows  {aligned.index.min():.2f}–{aligned.index.max():.2f}")
print(f"  SST:            {aligned['SST'].notna().sum()} samples")
print(f"  Sea Level:      {aligned['Sea Level'].notna().sum()} samples")
print(f"  ENSO:           {aligned['ENSO'].notna().sum()} samples")
print(f"  d13c_sg_trend:  {aligned['d13c_sg_trend'].notna().sum()} samples")
print(f"  d18o_sg_trend:  {aligned['d18o_pdb_sg_trend'].notna().sum()} samples")
aligned

In [ ]:
correlation_matrix = aligned.corr()

# Create a heatmap
fig = plt.figure(figsize=(8,5))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt=".2f")
plt.title("Correlation Matrix Heatmap")
plt.tight_layout()
plt.show()
#fig.savefig('corr_matrix.png')

In [ ]:
climate

In [ ]:
import matplotlib as mpl

# ENSO event windows for shading
_ENSO_WINDOWS = [
    (1982 + 2/12,  1984 + 1/12,  '#ff6b35'),   # El Niño
    (1986 + 9/12,  1988 + 1/12,  '#ff6b35'),
    (1992 + 1/12,  1993 + 1/12,  '#ff6b35'),
    (1997 + 2/12,  1998 + 10/12, '#ff6b35'),
    (1984 + 6/12,  1985 + 6/12,  '#00bfff'),   # La Niña
    (1988 + 6/12,  1989 + 6/12,  '#00bfff'),
    (1998 + 10/12, 2000 + 3/12,  '#00bfff'),
]

def plot_rolling_corr(df, proxy, drivers, win,
                      proxy_col='#27F535', driver_cols=None,
                      xlim=(1958, 2007), save_path=None):
    """
    Plot rolling correlation between `proxy` and each series in `drivers`.

    Parameters
    ----------
    df          : aligned DataFrame with float (decimal-year) index
    proxy       : column name of the coral proxy (e.g. 'd13c_no_tec_sg')
    drivers     : list of (col_name, label, colour) tuples
    win         : rolling window size in samples
    proxy_col   : colour for the proxy trace
    xlim        : (xmin, xmax) for the x-axis
    save_path   : if given, save figure to this path
    """
    if driver_cols is None:
        driver_cols = ['#4ecdc4', '#e05c5c', '#aaaaaa']

    n      = len(drivers)
    fig, axes = plt.subplots(
        n * 2, 1, figsize=(18, 4 * n),
        sharex=True,
        gridspec_kw={'hspace': 0,
                     'height_ratios': [2, 1] * n}
    )

    cmap_rc = mpl.cm.RdBu_r
    norm_rc  = mpl.colors.Normalize(vmin=-1, vmax=1)

    x = df.index.values

    for pi, (drv_col, drv_lbl, dc) in enumerate(drivers):
        ax_sig = axes[pi * 2]
        ax_cor = axes[pi * 2 + 1]

        for ax in (ax_sig, ax_cor):
            #ax.set_facecolor('#111720')
            for sp in ax.spines.values():
                sp.set_color('#1e2a38')
            ax.tick_params(colors='#5a7090', labelsize=8.5)
            ax.grid(color='#1a2535', lw=0.5, alpha=0.6)

        p  = df[proxy].values
        dr = df[drv_col].values

        # Signal panel
        ax_sig.plot(x, p, color=proxy_col, lw=0.8, alpha=0.35)
        ax_sig.plot(x,
                    pd.Series(p).rolling(8, center=True, min_periods=3).mean().values,
                    color=proxy_col, lw=2.2)
        ax_sig.set_ylabel(proxy, color=proxy_col, fontsize=8.5)
        ax_sig.tick_params(axis='y', labelcolor=proxy_col)

        ax2 = ax_sig.twinx()
        #ax2.set_facecolor('#111720')
        valid_d = ~np.isnan(dr)
        ax2.plot(x[valid_d], dr[valid_d], color=dc, lw=1.5, alpha=0.85)
        ax2.set_ylabel(drv_lbl, color=dc, fontsize=8.5)
        ax2.tick_params(axis='y', labelcolor=dc)
        for sp in ax2.spines.values():
            sp.set_color('#1e2a38')

        # Rolling correlation
        rc = df[proxy].rolling(win, center=True, min_periods=win // 2).corr(df[drv_col])
        y     = rc.values
        valid = ~np.isnan(y)
        xv    = x[valid]; yv = y[valid]

        ax_cor.plot(x, y, color='white', lw=1.0, zorder=3)
        ax_cor.axhline(0, color='#5a7090', lw=0.8, ls='--')
        for i in range(len(xv) - 1):
            ax_cor.axvspan(xv[i], xv[i+1],
                           color=cmap_rc(norm_rc(yv[i])), alpha=0.65, linewidth=0)

        n_eff  = int(df[drv_col].notna().sum())
        thresh = 1.97 / np.sqrt(max(min(win, n_eff) - 2, 2))
        ax_cor.axhline( thresh, color='white', lw=0.9, ls=':', alpha=0.5,
                        label=f'p=0.05 (±{thresh:.2f})')
        ax_cor.axhline(-thresh, color='white', lw=0.9, ls=':', alpha=0.5)
        ax_cor.set_ylim(-1, 1)
        ax_cor.set_ylabel('r', color='#8a9bb0', fontsize=8.5)
        if pi == 0:
            ax_cor.legend(fontsize=7.5, facecolor='#111720',
                          edgecolor='#1e2a38', labelcolor='#cdd6e0', loc='upper left')

        # ENSO shading
        for t0, t1, ec in _ENSO_WINDOWS:
            ax_sig.axvspan(t0, t1, alpha=0.08, color=ec, zorder=0)
            ax_cor.axvspan(t0, t1, alpha=0.08, color=ec, zorder=0)

    axes[-1].set_xlabel('Calendar year', color='#8a9bb0', fontsize=9.5)
    for ax in axes:
        ax.set_xlim(xlim)

    axes[0].set_title(
        f'Rolling Correlation — {proxy}  |  window = {win} samples ≈ {win/8:.1f} yr  |  '        f'orange = El Niño  |  blue = La Niña  |  dotted = p=0.05',
        color='#cdd6e0', fontsize=11, pad=8
    )
    #fig.patch.set_facecolor('#0a0e14')



In [ ]:
import matplotlib as mpl

# ENSO event windows for shading
_ENSO_WINDOWS = [
    (1982 + 2/12,  1984 + 1/12,  '#ff6b35'),
    (1986 + 9/12,  1988 + 1/12,  '#ff6b35'),
    (1992 + 1/12,  1993 + 1/12,  '#ff6b35'),
    (1997 + 2/12,  1998 + 10/12, '#ff6b35'),
    (1984 + 6/12,  1985 + 6/12,  '#00bfff'),
    (1988 + 6/12,  1989 + 6/12,  '#00bfff'),
    (1998 + 10/12, 2000 + 3/12,  '#00bfff'),
]

FS = 13  # global fontsize

def plot_rolling_corr(df, proxy, drivers, win,
                      proxy_col='#0BF41B', proxy_col2='#05700C',driver_cols=None,
                      xlim=(1958, 2007), save_path=None):
    if driver_cols is None:
        driver_cols = ['#4ecdc4', '#e05c5c', '#aaaaaa']

    n = len(drivers)
    fig, axes = plt.subplots(
        n * 2, 1, figsize=(12, 4 * n),
        sharex=True,
        gridspec_kw={'hspace': 0,
                     'height_ratios': [2, 1] * n}
    )

    cmap_rc = mpl.cm.RdBu_r
    norm_rc  = mpl.colors.Normalize(vmin=-1, vmax=1)

    x = df.index.values

    for pi, (drv_col, drv_lbl, dc) in enumerate(drivers):
        ax_sig = axes[pi * 2]
        ax_cor = axes[pi * 2 + 1]

        for ax in (ax_sig, ax_cor):
            ax.set_facecolor('white')
            for sp in ax.spines.values():
                sp.set_color('k')
            ax.tick_params(colors='black', labelsize=FS)
            ax.grid(color='#e0e0e0', lw=0.5, alpha=0.8)

        p  = df[proxy].values
        #p2 = df[proxy2].values
        dr = df[drv_col].values

        # Signal panel
        ax_sig.plot(x, p, color=proxy_col, lw=0.8, alpha=1, label='raw')
        #ax_sig.plot(x, p2, color=proxy_col2, lw=2.2, label='filtered')
        ax_sig.plot(x,
            pd.Series(p).rolling(8, center=True, min_periods=3).mean().values,
            color=proxy_col2, lw=2.2)
        ax_sig.set_ylabel(r'$\delta^{13}$C', color=proxy_col, fontsize=FS)
        ax_sig.tick_params(axis='y', labelcolor=proxy_col, labelsize=FS)

        ax2 = ax_sig.twinx()
        ax2.set_facecolor('white')
        valid_d = ~np.isnan(dr)
        ax2.plot(x[valid_d], dr[valid_d], color=dc, lw=1.5, alpha=1)
        ax2.set_ylabel(drv_lbl, color=dc, fontsize=FS)
        ax2.tick_params(axis='y', labelcolor=dc, labelsize=FS)
        for sp in ax2.spines.values():
            sp.set_color('k')

        # Rolling correlation
        rc = df[proxy].rolling(win, center=True, min_periods=win // 2).corr(df[drv_col])
        y     = rc.values
        valid = ~np.isnan(y)
        xv    = x[valid]; yv = y[valid]

        ax_cor.plot(x, y, color='black', lw=1.0, zorder=3)
        ax_cor.axhline(0, color='#888888', lw=0.8, ls='--')
        for i in range(len(xv) - 1):
            ax_cor.axvspan(xv[i], xv[i+1],
                           color=cmap_rc(norm_rc(yv[i])), alpha=0.55, linewidth=0)

        n_eff  = int(df[drv_col].notna().sum())
        thresh = 1.97 / np.sqrt(max(min(win, n_eff) - 2, 2))
        ax_cor.axhline( thresh, color='black', lw=0.9, ls=':', alpha=0.7,
                        label=f'p=0.05 (±{thresh:.2f})')
        ax_cor.axhline(-thresh, color='black', lw=0.9, ls=':', alpha=0.7)
        ax_cor.set_ylim(-1, 1)
        ax_cor.set_ylabel('r', color='black', fontsize=FS)
        ax_cor.tick_params(axis='y', labelsize=FS)
        if pi == 0:
            ax_cor.legend(fontsize=FS, facecolor='white',
                          edgecolor='k', labelcolor='black', loc='upper left')

        # ENSO shading
        for t0, t1, ec in _ENSO_WINDOWS:
            ax_sig.axvspan(t0, t1, alpha=0.10, color=ec, zorder=0)
            ax_cor.axvspan(t0, t1, alpha=0.10, color=ec, zorder=0)

    # Bottom x-axis label
    axes[-1].set_xlabel('Year', color='black', fontsize=FS)
    axes[-1].tick_params(axis='x', labelsize=FS)
    for ax in axes:
        ax.set_xlim(xlim)

    # Top-axis year ticks on the first (topmost) signal panel
    ax_top = axes[0].twiny()
    ax_top.set_xlim(xlim)
    ax_top.tick_params(axis='x', which='both', direction='in',
                       labelsize=FS, colors='black', top=True)
    ax_top.set_xlabel('Year', fontsize=FS, labelpad=6)
    # Match tick positions to bottom axis so they line up
    ax_top.set_xticks(axes[-1].get_xticks())
    ax_top.set_xticklabels([str(int(t)) for t in axes[-1].get_xticks()])
    for sp in ax_top.spines.values():
        sp.set_visible(False)
    ax_top.spines['top'].set_visible(True)
    ax_top.spines['top'].set_color('k')

    axes[0].set_title(
        f'Rolling Correlation — {proxy}  |  window = {win} samples ≈ {win/8:.1f} yr  |  '
        f'orange = El Niño  |  blue = La Niña  |  dotted = p=0.05',
        color='black', fontsize=FS, pad=28   # extra pad to clear the top-axis labels
    )

    fig.patch.set_facecolor('white')

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

In [ ]:
aligned

In [ ]:
# δ¹³C_no_tec_sg vs climate / sea level / ENSO  — two window sizes
drivers_d13c = [
    ('synthetic_d18o', r'Climate based $\delta^{18}$O',  '#2EA39B'),
    ('Sea Level',      'Honoraria Tide Gauge [m]',   '#B02121'),
    ('ENSO',          'ENSO index',       '#3B3B3B'),
]

#for win in [16, 36]:
#    plot_rolling_corr(aligned, 'd13c_no_tec_sg', drivers_d13c, win=win,
#                      save_path=f'correlation_d13c_win{win}.png')

win =16
plot_rolling_corr(aligned, 'd13c_no_tec_sg', drivers_d13c, win=win, save_path=f'correlation_d13c_win{win}.pdf')



In [ ]:
# δ¹³C_no_tec_sg vs climate / sea level / ENSO  — two window sizes
drivers_d18o = [
    ('synthetic_d18o', r'Climate δ¹⁸O',  '#4ecdc4'),
    ('Sea Level',      'Sea Level (m)',   '#e05c5c'),
    ('ENSO',          'ENSO index',       '#aaaaaa'),
]

for win in [16, 36]:
    plot_rolling_corr(aligned, 'd18o_pdb_no_tec_sg', drivers_d18o, win=win,
                      save_path=f'correlation_d13c_win{win}.png')

In [ ]:
corr_win_size = 16


fig, axes = plt.subplots( 6,1 , figsize=(15,12), sharex=True, gridspec_kw={'hspace':0})

rolling_corr_1 = aligned['d13c_no_tec_sg'].rolling(window=corr_win_size, center=True).corr(aligned['SST'])
rolling_corr_age_1 = [ datetime_to_decimal_year(rolling_corr_1.index[i]) for i in range(len(rolling_corr_1))]

rolling_corr_2 = aligned['d13c_no_tec_sg'].rolling(window=corr_win_size, center=True).corr(aligned['Sea Level'])
rolling_corr_age_2 = [ datetime_to_decimal_year(rolling_corr_2.index[i]) for i in range(len(rolling_corr_2))]

rolling_corr_3 = aligned['d13c_no_tec_sg'].rolling(window=corr_win_size, center=True).corr(aligned['ENSO'])
rolling_corr_age_3 = [ datetime_to_decimal_year(rolling_corr_3.index[i]) for i in range(len(rolling_corr_3))]

# --- create diverging colormap ---
cmap = plt.cm.RdBu_r     # blue = positive, red = negative
norm = mpl.colors.Normalize(vmin=-1, vmax=1)


#-----------------------------------------------------------------------------

ax = axes[0]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d13c_no_tec_sg'], color='#27F535')
ax.set_ylabel(r'$\delta^{13}$C Observed'+'\n'+'detrended')


ax2= ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['SST'], color='red')
ax2.set_ylabel('SST [°C]')


ax = axes[1]

x = rolling_corr_age_1
y = rolling_corr_1.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)

# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)


#-----------------------------------------------------------------------------------

ax = axes[2]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d13c_no_tec_sg'], color='#27F535')
ax.set_ylabel(r'$\delta^{13}$C Observed'+'\n'+'detrended')


ax2 = ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['Sea Level'], color='red')
ax2.set_ylabel('Sea Level [m]')

ax = axes[3]

x = rolling_corr_age_2
y = rolling_corr_2.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)



# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)

#-----------------------------------------------------------------------------------

ax = axes[4]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d13c_no_tec_sg'], color='#27F535')
ax.set_ylabel(r'$\delta^{13}$C Observed'+'\n'+'detrended')


ax2 = ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['ENSO'], color='k')
ax2.set_ylabel(r'ENSO Index')


ax = axes[5]

x = rolling_corr_age_3
y = rolling_corr_3.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)

# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)
axes[0].set_title('Rolling Correlation Window Size: {:}'.format(corr_win_size),fontsize=20)

#fig.savefig('correlation_d13c.png')

In [ ]:
aligned_cut = aligned.loc[:'2002-08-01']

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

ax = axes[0]
ax.plot(aligned_cut['d13c_no_tectonic'], aligned_cut['SST'] , '.')



ax = axes[1]
ax.plot(aligned_cut['d13c_no_tectonic'], aligned_cut['Sea Level'], '.')




ax = axes[2]
ax.plot(aligned_cut['d13c_no_tectonic'], aligned_cut['ENSO'], '.' )


test_proxy = aligned_cut['Sea Level']/aligned_cut['Sea Level'].mean() +  aligned_cut['SST']/aligned_cut['SST'].mean()

ax = axes[3]
ax.plot(aligned_cut['d13c_no_tectonic'], test_proxy, '.')

In [ ]:
corr_win_size=36

fig, axes = plt.subplots( 6,1 , figsize=(15,12), sharex=True, gridspec_kw={'hspace':0})

rolling_corr_1 = aligned['d18o'].rolling(window=corr_win_size, center=True).corr(aligned['synthetic_d18o'])
rolling_corr_age_1 = [ datetime_to_decimal_year(rolling_corr_1.index[i]) for i in range(len(rolling_corr_1))]

rolling_corr_2 = aligned['d18o'].rolling(window=corr_win_size, center=True).corr(aligned['Sea Level'])
rolling_corr_age_2 = [ datetime_to_decimal_year(rolling_corr_2.index[i]) for i in range(len(rolling_corr_2))]

rolling_corr_3 = aligned['d18o'].rolling(window=corr_win_size, center=True).corr(aligned['ENSO'])
rolling_corr_age_3 = [ datetime_to_decimal_year(rolling_corr_3.index[i]) for i in range(len(rolling_corr_3))]

# --- create diverging colormap ---
cmap = plt.cm.RdBu_r     # blue = positive, red = negative
norm = mpl.colors.Normalize(vmin=-1, vmax=1)


#-----------------------------------------------------------------------------

ax = axes[0]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d18o'], color='#3492eb')
ax.set_ylabel(r'$\delta^{18}$O Observed')


ax2= ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['synthetic_d18o'], color='blue')
ax2.set_ylabel(r'Climate Based $\delta^{18}$O')


ax = axes[1]

x = rolling_corr_age_1
y = rolling_corr_1.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)

# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)


#-----------------------------------------------------------------------------------

ax = axes[2]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d18o'], color='#3492eb')
ax.set_ylabel(r'$\delta^{18}$O Observed')


ax2 = ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['Sea Level'], color='red')
ax2.set_ylabel('Sea Level [m]')

ax = axes[3]

x = rolling_corr_age_2
y = rolling_corr_2.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)



# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)

#-----------------------------------------------------------------------------------

ax = axes[4]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d18o'], color='#3492eb')
ax.set_ylabel(r'$\delta^{18}$O Observed')


ax2 = ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['ENSO'], color='k')
ax2.set_ylabel(r'ENSO Index')


ax = axes[5]

x = rolling_corr_age_3
y = rolling_corr_3.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)

# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)

#fig.suptitle('Rolling Correlation Window Size: {:}'.format(corr_win_size))
axes[0].set_title('Rolling Correlation Window Size: {:}'.format(corr_win_size),fontsize=20)

fig.savefig('correlation_d18o.png')

In [ ]:
corr_win_size=36

fig, axes = plt.subplots( 6,1 , figsize=(15,12), sharex=True, gridspec_kw={'hspace':0})

rolling_corr_1 = aligned['d18o_no_tec_sg'].rolling(window=corr_win_size, center=True).corr(aligned['synthetic_d18o'])
rolling_corr_age_1 = [ datetime_to_decimal_year(rolling_corr_1.index[i]) for i in range(len(rolling_corr_1))]

rolling_corr_2 = aligned['d18o_no_tec_sg'].rolling(window=corr_win_size, center=True).corr(aligned['Sea Level'])
rolling_corr_age_2 = [ datetime_to_decimal_year(rolling_corr_2.index[i]) for i in range(len(rolling_corr_2))]

rolling_corr_3 = aligned['d18o_no_tec_sg'].rolling(window=corr_win_size, center=True).corr(aligned['ENSO'])
rolling_corr_age_3 = [ datetime_to_decimal_year(rolling_corr_3.index[i]) for i in range(len(rolling_corr_3))]

# --- create diverging colormap ---
cmap = plt.cm.RdBu_r     # blue = positive, red = negative
norm = mpl.colors.Normalize(vmin=-1, vmax=1)


#-----------------------------------------------------------------------------

ax = axes[0]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d18o_no_tec_sg'], color='#3492eb')
ax.set_ylabel(r'$\delta^{18}$O Observed')


ax2= ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['synthetic_d18o'], color='blue')
ax2.set_ylabel(r'Climate Based $\delta^{18}$O')


ax = axes[1]

x = rolling_corr_age_1
y = rolling_corr_1.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)

# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)


#-----------------------------------------------------------------------------------

ax = axes[2]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d18o_no_tec_sg'], color='#3492eb')
ax.set_ylabel(r'$\delta^{18}$O Observed')


ax2 = ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['Sea Level'], color='red')
ax2.set_ylabel('Sea Level [m]')

ax = axes[3]

x = rolling_corr_age_2
y = rolling_corr_2.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)



# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)

#-----------------------------------------------------------------------------------

ax = axes[4]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d18o_no_tec_sg'], color='#3492eb')
ax.set_ylabel(r'$\delta^{18}$O Observed')


ax2 = ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['ENSO'], color='k')
ax2.set_ylabel(r'ENSO Index')


ax = axes[5]

x = rolling_corr_age_3
y = rolling_corr_3.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)

# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)

#fig.suptitle('Rolling Correlation Window Size: {:}'.format(corr_win_size))
axes[0].set_title('Rolling Correlation Window Size: {:}'.format(corr_win_size),fontsize=20)

#fig.savefig('correlation_d18o.png')

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="markers", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)
fig.add_trace(go.Scatter(x=morpho_memo['age_final'], y=morpho_memo['y_true'], mode="lines", line=dict(color='purple', width=4),text=morpho_memo['sample _name'], name='SPLA morpho memo'), secondary_y=True)


fig.add_trace(go.Scatter(x=morpho_data['date']-0.5, y=-morpho_data['SPLA_luc'], mode="lines", line=dict(color='firebrick', width=4), name='SPLA morpho luc'), secondary_y=True)
fig.add_trace(go.Scatter(x=morpho_data['date']-0.5, y=morpho_data['SPLA_fred'], mode="lines", line=dict(color='orange', width=4), name='SPLA morpho fred'), secondary_y=True)

fig.add_trace(go.Scatter(x=morpho_data['date']-0.5, y=-morpho_data['SPA_luc'], mode="lines", line=dict(color='navy', width=4), name='SPA morpho luc'), secondary_y=True)
fig.add_trace(go.Scatter(x=morpho_data['date']-0.5, y=-morpho_data['SPA_fred'], mode="lines", line=dict(color='black', width=4), name='SPA morpho fred'), secondary_y=True)




fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1000, height=600, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= 'Sea Level [mm]', secondary_y=True)

fig.show()
#fig.write_html('d13c_all-morphologies.html')


In [ ]:
fred

In [ ]:
 fred.loc[90:,['calendar_year','d13c','d18o','d18o_pdb']].rolling(window_size1, closed='left').mean().shift(-(window_size1 ))

In [ ]:
# assign
fred.loc[:89, ['age_ma2','d13c_ma2','d18o_ma2','d18o_pdb_ma2']] = right_roll.loc[:89].to_numpy()
fred.loc[90:, ['age_ma2','d13c_ma2','d18o_ma2','d18o_pdb_ma2']] = left_roll.loc[90:].to_numpy()

In [ ]:
#fred[fred['decimal_year']<2007.25]
x = pd.concat([fred.loc[:89,['d13c','d18o','calendar_year','d18o_pdb']].rolling(window_size1, center=True).mean(),
         fred.loc[90:,['d13c','d18o','calendar_year','d18o_pdb']].rolling(window_size1).mean().shift(-(window_size1//2 ))])


x.head(20)

In [ ]:
right_roll.loc[:89].to_numpy()

In [ ]:
window_size1 = 16
split = 2007.25

# split masks
before = fred["calendar_year"] <= split
after = fred["calendar_year"] > split

# rolling means
right_roll = fred["d13c"].rolling(window_size1).mean()
left_roll = fred["d13c"].rolling(window_size1).mean().shift(-(window_size1-1))

ma = pd.Series(index=fred.index, dtype=float)
ma.loc[before] = right_roll.loc[before]
ma.loc[after] = left_roll.loc[after]

In [ ]:

ma

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d13c'] , mode="markers", line_color='#27F535', name='fred d13c'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['age_ma'], y=fred['d13c_ma'] , mode="lines", line_color='#05700C', name='moving average'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['calendar_year'], y=ma , mode="lines", line_color='#011803', name='moving average 2'), secondary_y=False)


fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d18o'] , mode="markers", line_color='#3492eb', name='fred d18O'), secondary_y=True)
fig.add_trace(go.Scatter(x=fred['age_ma'], y=fred['d18o_ma'] , mode="lines", line_color='#134b80', name='moving average'), secondary_y=True)

fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1600, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ C Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=True)


fig.show()
#fig.write_html('d13c_all-morphologies.html')


In [ ]:
window_size1=12

fred_old = pd.read_excel('../../../2025-08-22_FredsData-post2008/FredData_editedMemo_raw.xlsx').dropna()
fred_old['d18o_pdb'] = -(fred['d18o']-20.91)/1.03091
fred_old[['age_ma','d13c_ma','d18o_ma', 'd18o_pdb_ma']] = fred[['calendar_year','d13c','d18o','d18o_pdb']].rolling(window_size, center=True).mean()
fred_old

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d13c'] , mode="markers", line_color='#27F535', name='fred d13c'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['age_ma'], y=fred['d13c_ma'] , mode="lines", line_color='#05700C', name='moving average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=fred['calendar_year'], y=ma , mode="lines", line_color='#011803', name='moving average 2'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred_old['decimal_year'], y=fred['d13c'] , mode="markers", line_color='#F54522', name='old age model'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred_old['age_ma'], y=fred['d13c_ma'] , mode="lines", line_color='#872109', name='moving average'), secondary_y=False)



#fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d18o'] , mode="markers", line_color='#3492eb', name='fred d18O'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['age_ma'], y=fred['d18o_ma'] , mode="lines", line_color='#134b80', name='moving average'), secondary_y=True)

fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1600, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ C Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=True)


fig.show()
#fig.write_html('d13c_all-morphologies.html')
